### Week 6, Day 2

自分たちの MCP サーバーを作る前に、世の中にどんなものがあるか、人気のマーケットプレイスを2つ見てみましょう。

https://glama.ai/mcp  
https://smithery.ai/servers 

いよいよ自分たちの MCP サーバーを作って使ってみます!

とてもシンプルですが、超シンプルというわけでもありません。MCP をめぐる盛り上がりは、他の人が作った MCP サーバーを共有したり使ったりすることがいかに簡単か、という点にありますが、自分で作るとなると、それなりの作業が必要になります。

まずは、勤勉なエンジニアリングチームが大部分を作ってくれた Python コードを見てみましょう。

backend/accounts.py

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

In [ ]:
# メインモデルをOpenAIからGeminiに切り替える
import os
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(api_key=os.getenv("GOOGLE_API_KEY"), base_url=GEMINI_BASE_URL)
MODEL_NAME = OpenAIChatCompletionsModel(model="gemini-flash-latest", openai_client=gemini_client)

# GeminiのキーではOpenAIのトレース機能(platform.openai.com/traces)は使えないため無効化
set_tracing_disabled(True)


In [ ]:
# Windows では、Jupyter カーネルから起動した stdio MCP サーバーが、実際のファイルディスクリプタを
# 持たない stderr ストリームに書き込もうとして io.UnsupportedOperation: fileno でクラッシュする。
# サーバーの stderr をヌルデバイスに送ることで、常に書き込める実在の場所を用意し、これにより
# 以降のすべてのセルで OpenAI Agents SDK のドキュメントどおりに MCPServerStdio を使えるようにする。
# Mac と Linux では影響がない。
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

## この Account という Python モジュールがどこから来たか、想像がつきますか?!

私が書いたものではありません!

In [ ]:
from backend.accounts import Account

In [ ]:
account = Account.get("Ed")
account.reset()
account

In [ ]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

In [ ]:
account.report()

In [ ]:
account.list_transactions()

### さあ、MCP サーバーを書いて、直接使ってみましょう!

In [ ]:
# では、自分たちの accounts サーバーを MCP サーバーとして使ってみましょう

params = {"command": "uv", "args": ["run", "-m", "backend.accounts_server"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [ ]:
mcp_tools

In [ ]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
model = MODEL_NAME

In [ ]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">エクササイズ</h2>
            <span style="color:#ff7800;">自分だけの MCP サーバーを作ってみましょう! プッシュ通知を送るシンプルな関数を作って、
            その結果を楽しんでください! ヒントが必要な場合は backend/push_server.py に解答例があります。
            </span>
        </td>
    </tr>
</table>